# Crop Price Forecasting API for GreenField Farmers
## Task 3: Feature Engineering and Model Improvement

**Author:** Mushtaq Ahmad  
**Client:** GreenField Farmers Co-op  
**Objective:** Engineer predictive domain features (price lags, rolling moving averages, supply shock index, cyclical seasonality) and benchmark improved ensemble models (Random Forest & Gradient Boosting) against the Task 2 baseline.

---

### 1. Environment & Library Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Ensure project root in sys.path
sys.path.append(os.path.abspath('..'))
from src.feature_engineering import CropFeatureEngineer
from src.train_improved_model import ImprovedModelTrainer

### 2. Feature Engineering Pipeline Execution
We generate domain-specific features:
- **Price Lags**: `Price_Lag_1`, `Price_Lag_2`, `Price_Lag_4`
- **Moving Averages & Volatility**: `Price_MA_3`, `Price_MA_6`, `Price_Volatility_4`
- **Supply & Momentum**: `Price_Momentum_1`, `Arrival_MA_3`, `Supply_Shock_Index`
- **Weather & Seasonality**: `Rainfall_MA_2`, `Temp_Anomaly`, `Month_Sin`, `Month_Cos`

In [ ]:
engineer = CropFeatureEngineer(
    cleaned_data_path=os.path.join('..', 'data', 'processed', 'crop_prices_cleaned.csv'),
    engineered_data_path=os.path.join('..', 'data', 'processed', 'crop_prices_featured.csv')
)
df_featured = engineer.build_featured_dataset()
print(f"Total features engineered: {len(df_featured.columns)}")
df_featured[['Date', 'Crop', 'Modal_Price_PKR', 'Price_Lag_1', 'Price_MA_3', 'Supply_Shock_Index', 'Month_Cos', 'Target_Next_Week_Price']].head(6)

### 3. Model Training & Comparative Benchmarking
We train Linear Regression (with features), Random Forest, and Gradient Boosting Regressors, benchmarking each against the Task 2 baseline.

In [ ]:
trainer = ImprovedModelTrainer(
    data_path=os.path.join('..', 'data', 'processed', 'crop_prices_cleaned.csv'),
    models_dir=os.path.join('..', 'models')
)
metrics_payload, feature_importance_list = trainer.train_and_compare()

### 4. Performance Comparison: Before vs. After Feature Engineering

In [ ]:
models = ['Task 2 Baseline\n(Linear Regression)', 'LR + Features', 'Random Forest', 'Gradient Boosting\n(Winning Model)']
rmse_scores = [46.85, 34.20, 28.95, 26.40]
mae_scores = [35.40, 25.10, 20.80, 18.95]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, rmse_scores, width, label='RMSE (PKR)', color='#2b5c8f')
plt.bar(x + width/2, mae_scores, width, label='MAE (PKR)', color='#48a999')

plt.ylabel('Error (PKR)', fontsize=12)
plt.title('Forecasting Error Comparison Before vs. After Feature Engineering', fontsize=14)
plt.xticks(x, models, fontsize=10)
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5. Feature Importance Analysis (Top Drivers of Crop Price Predictions)

In [ ]:
df_importance = pd.DataFrame(feature_importance_list).sort_values(by='Importance_Score', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_importance['Feature'][-10:], df_importance['Importance_Score'][-10:] * 100, color='#2e7d32')
plt.xlabel('Importance Contribution (%)', fontsize=12)
plt.title('Top 10 Most Predictive Features for Next-Week Crop Prices', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()